# Process and Save top NER keyword

        Count top keywords for each NE 
        Name	Description
        CARDINAL	數字
        DATE	日期
        EVENT	事件
        FAC	設施
        GPE	行政區
        LANGUAGE	語言
        LAW	法律
        LOC	地理區
        MONEY	金錢
        NORP	民族、宗教、政治團體
        ORDINAL	序數
        ORG	組織
        PERCENT	百分比率
        PERSON	人物
        PRODUCT	產品
        QUANTITY	數量
        TIME	時間
        WORK_OF_ART	作品

        The popular keywords for each NE and Category.
        This table is what we want to get. 
                ne_name	top_keys
        0	CARDINAL[(政治, [('2000', 7), ('4000', 2), ('13', 2), ('...
        1	DATE	[(政治, [('今天', 35), ('12天', 2), ('5月', 2), ('同年...
        2	EVENT	[(政治, [('COVID-19（2019冠狀病毒疾病）疫情', 1), ('俄烏戰爭',...
        3	FAC	[(政治, [('台東志航基地', 8), ('台東馬偕醫院', 5), ('新竹空軍基地'...
        4	GPE	[(政治, [('台灣', 25), ('台東', 10), ('法國', 8), ('烏克...
        5	LANGUAGE	[(政治, [('英文', 1)]), (科技, []), (運動, [('中文', 1)]...
        6	LAW	[(政治, []), (科技, [('LINE', 1)]), (運動, []), (證卷,...
        7	LOC	[(政治, [('台灣西南防空識別區', 1), ('北美', 1), ('屯稜線', 1)...
        8	MONEY	[(政治, [('新台幣6億元', 1), ('3萬5703美元', 1), ('約101萬...
        9	NORP	[(政治, [('烏克蘭', 6), ('德商', 2), ('歐商', 1), ('美商'...
        10	ORDINAL	[(政治, [('第四', 2), ('第1', 1), ('第206', 1), ('第8...
        11	ORG	[(政治, [('外交部', 16), ('立法院', 13), ('監察院', 9), (...
        12	PERCENT	[(政治, [('5%', 1), ('45%', 1)]), (科技, [('100%',...
        13	PERSON	[(政治, [('黃重凱', 24), ('柳惠千', 13), ('喬建中', 12), ...
        14	PRODUCT	[(政治, [('2000', 3), ('殲16戰機', 2), ('2017', 1),...
        15	QUANTITY	[(政治, [('2000呎', 6), ('10浬', 5), ('80公頃', 2), ...
        16	TIME	[(政治, [('下午', 10), ('上午', 10), ('11時26分', 5), ...
        17	WORK_OF_ART	[(政治, []), (科技, []), (運動, []), (證卷, []), (產經, ...

# Put them together

In [1]:
import pandas as pd
from collections import Counter

# Read data
df = pd.read_csv('cna_news_preprocessed.csv',sep='|')

# NerToken(word='烏克蘭', ner='GPE', idx=(4, 7))  # call function NerToken with three parameters: word, ner, and idx
# We need the name and keyword
def NerToken(word, ner, idx):
    # print(ner,word)
    return ner,word

# Count top-200 hot keywords for each NER name
# 統計各NER的熱門關鍵字
# word count for a piece of news
def ne_word_frequency( a_news_ne ):
    filtered_words =[]
    for ner,word in a_news_ne:
        if (len(word) >= 2) & (ner in allowedNE):
            filtered_words.append(word)
    counter = Counter( filtered_words )
    return counter.most_common( 20 )

## Count top 200 word frequency for each category given a certain NE, e.g., "PERSON"
'''
    {'政治': [('黃重凱', 24),
    ('柳惠千', 13),
    ('喬建中', 12),
    ('徐國勇', 11),
    ('季欽', 8),
    ('許舒博', 7),
    ('游錫堃', 6),

'''
news_links =['aipl', 'ait', 'aspt', 'asc', 'aie', 'amov','ahel','aopl','asoc','acul','acn']
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']

## To conveniently save data using pandas, we should convert dict to list.
# All-in-one function
def get_top_ner_words():
    top_cate_ner_words={}
    words_all=[]
    for category in news_categories:
        df_group = df[df.category == category]
        words_group = []

        # concatenate terms in a category
        for row in df_group.entities:
            words_group += eval(row)

        # concatenate all terms
        words_all += words_group

        # Get top words by calling ne_word_frequency() function
        topwords = ne_word_frequency( words_group )
        top_cate_ner_words[category] = topwords

    topwords_all = ne_word_frequency(words_all)
    top_cate_ner_words['全部'] = topwords_all
    
    return list(top_cate_ner_words.items()) # convert to list
    # return top_cate_ne_words

# (1) 熱門人物
## Save top-200 hot persons for future usage
# We already have done this in our previous app.
## The most popular products
#allowedNE=['PRODUCT']
allowedNE=['PERSON']
hotPersons = get_top_ner_words()
df_hotPersons = pd.DataFrame(hotPersons, columns = ['category','top_keys'])
df_hotPersons.to_csv('news_top_person_by_category_via_ner.csv', sep=',', index=False) # 與上周的任務相同


# (2) 熱門關鍵字
## We can get top 200 word frequecy for each news categroy by using "entities" column.
allowedNE=['EVENT','FAC','GPE','LANGUAGE','LAW','LOC','NORP','ORG','PERSON','PRODUCT','WORK_OF_ART']
top_group_ner_words = get_top_ner_words()
df_top_group_ner_words = pd.DataFrame(top_group_ner_words, columns = ['category','top_keys'])
df_top_group_ner_words.to_csv('news_topkey_with_category_via_ner.csv', sep=',', index=False) # 與第一次的Django任務相同

# (3) 每個NER的熱門關鍵字
# Count top-200 hot keywords for each NE name
# NER詞性
NE_Name=['CARDINAL','DATE','EVENT','FAC','GPE','LANGUAGE','LAW','LOC','MONEY','NORP','ORDINAL','ORG','PERCENT','PERSON','PRODUCT','QUANTITY','TIME','WORK_OF_ART']
# It takes time, at least 2 minitues.
top_word_NE=[]
for ne_name in NE_Name:
    allowedNE= [ne_name]
    topwords = get_top_ner_words()
    top_word_NE.append([ne_name, topwords])
## Save each ne's top 200 category word frequency
df_top_word_NE = pd.DataFrame(top_word_NE, columns = ['ne_name','top_keys'])
df_top_word_NE.to_csv('news_topkey_by_ner_and_category.csv', sep=',', index=False) # 本次的NER任務

# Step by step demonstration

# Read data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('cna_news_preprocessed.csv',sep='|')

In [3]:
df.head(1)

,item_id,date,category,title,content,sentiment,summary,top_key_freq,tokens,tokens_v2,entities,token_pos,link,photo_link
0,aipl_20220314_1,2022/03/14 21:38,政治,外交部援烏物資已募4000箱 吳釗燮感謝捐贈民眾,民眾捐贈烏克蘭的愛心物資持續湧入外交部，截至今天傍晚累計已收到約4000箱，外交部長吳釗燮中...,0.01,"['外交部除感謝熱心民眾踴躍捐贈援助烏克蘭人道物資外', '親赴外交部捐贈物資的民眾約173...","[('外交部', 14), ('民眾', 7), ('物資', 7), ('烏克蘭', 5)...","['民眾', '捐贈', '烏克蘭', '的', '愛心', '物資', '持續', '湧入...","['民眾', '烏克蘭', '愛心', '物資', '外交部', '收到', '外交部長',...","[NerToken(word='烏克蘭', ner='GPE', idx=(4, 7)), ...","[('民眾', 'Na'), ('捐贈', 'VD'), ('烏克蘭', 'Nc'), ('...",https://www.cna.com.tw/news/aipl/202203140364....,https://imgcdn.cna.com.tw/www/WebPhotos/200/20...


# Take a look at the named entities

In [4]:
df.entities[0]

"[NerToken(word='烏克蘭', ner='GPE', idx=(4, 7)), NerToken(word='外交部', ner='ORG', idx=(16, 19)), NerToken(word='傍晚', ner='TIME', idx=(24, 26)), NerToken(word='4000', ner='CARDINAL', idx=(32, 36)), NerToken(word='外交部長', ner='ORG', idx=(38, 42)), NerToken(word='吳釗燮', ner='PERSON', idx=(42, 45)), NerToken(word='中午', ner='TIME', idx=(45, 47)), NerToken(word='外交部', ner='ORG', idx=(76, 79)), NerToken(word='晚間', ner='TIME', idx=(79, 81)), NerToken(word='外交部', ner='ORG', idx=(89, 92)), NerToken(word='7日', ner='DATE', idx=(93, 95)), NerToken(word='烏克蘭', ner='NORP', idx=(104, 107)), NerToken(word='外交部', ner='ORG', idx=(122, 125)), NerToken(word='1730', ner='CARDINAL', idx=(133, 137)), NerToken(word='4000', ner='CARDINAL', idx=(152, 156)), NerToken(word='18日', ner='DATE', idx=(195, 198)), NerToken(word='外交部', ner='ORG', idx=(199, 202)), NerToken(word='外交部', ner='ORG', idx=(228, 231)), NerToken(word='吳釗燮', ner='PERSON', idx=(234, 237)), NerToken(word='今天', ner='DATE', idx=(237, 239)), NerToken(word='

In [5]:
# NerToken(word='烏克蘭', ner='GPE', idx=(4, 7))  # call function NerToken with three parameters: word, ner, and idx
def NerToken(word, ner, idx):
    # print(ner,word)
    return ner,word


In [6]:
# call function NerToken with three parameters: word, ner, and idx
NerToken(word='烏克蘭', ner='GPE', idx=(4, 7))

('GPE', '烏克蘭')

In [7]:
eval(df.entities[0])

[('GPE', '烏克蘭'),
 ('ORG', '外交部'),
 ('TIME', '傍晚'),
 ('CARDINAL', '4000'),
 ('ORG', '外交部長'),
 ('PERSON', '吳釗燮'),
 ('TIME', '中午'),
 ('ORG', '外交部'),
 ('TIME', '晚間'),
 ('ORG', '外交部'),
 ('DATE', '7日'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('CARDINAL', '1730'),
 ('CARDINAL', '4000'),
 ('DATE', '18日'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('PERSON', '吳釗燮'),
 ('DATE', '今天'),
 ('TIME', '中午'),
 ('ORG', '外交部'),
 ('PERSON', '吳釗燮'),
 ('ORG', '慈濟'),
 ('NORP', '烏克蘭'),
 ('PERSON', '吳釗燮'),
 ('ORG', '外交部'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('DATE', '6個月'),
 ('DATE', '3月18日'),
 ('TIME', '下午'),
 ('TIME', '5時'),
 ('CARDINAL', '20'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部')]

In [9]:
# We need the name and keyword
for ner,key in eval(df.entities[0]):
    print(ner,key)

GPE 烏克蘭
ORG 外交部
TIME 傍晚
CARDINAL 4000
ORG 外交部長
PERSON 吳釗燮
TIME 中午
ORG 外交部
TIME 晚間
ORG 外交部
DATE 7日
NORP 烏克蘭
ORG 外交部
CARDINAL 1730
CARDINAL 4000
DATE 18日
ORG 外交部
ORG 外交部
PERSON 吳釗燮
DATE 今天
TIME 中午
ORG 外交部
PERSON 吳釗燮
ORG 慈濟
NORP 烏克蘭
PERSON 吳釗燮
ORG 外交部
NORP 烏克蘭
ORG 外交部
ORG 外交部
ORG 外交部
DATE 6個月
DATE 3月18日
TIME 下午
TIME 5時
CARDINAL 20
ORG 外交部
ORG 外交部
NORP 烏克蘭
ORG 外交部


# Count top-200 hot keywords for each NER name

統計各NER的熱門關鍵字

In [8]:
from collections import Counter

In [9]:
# Filter condition: two words and specified NER
# allowedNE=['EVENT','FAC','GPE','LANGUAGE','LAW','LOC','NORP','ORG','PERSON','PRODUCT','WORK_OF_ART']
allowedNE=['PERSON']

In [10]:
filtered_words =[]
for ner,word in eval(df.entities[0]):
    if (len(word) >= 2) & (ner in allowedNE):
        filtered_words.append(word)
counter = Counter( filtered_words )
counter.most_common( 50 )

[('吳釗燮', 4)]

In [11]:
# word count for a piece of news
def ne_word_frequency( a_news_ne ):
    filtered_words =[]
    for ner,word in a_news_ne:
        if (len(word) >= 2) & (ner in allowedNE):
            filtered_words.append(word)
    counter = Counter( filtered_words )
    return counter.most_common( 20 )

In [12]:
ne_word_frequency(eval(df.entities[0]))

[('吳釗燮', 4)]

In [13]:
ne_word_frequency(eval(df.entities[1]))

[('許邁德', 2), ('黃重凱', 1), ('柳惠千', 1), ('朱冠甍', 1)]

### Concatenate two lists

In [14]:
a=[]
a += eval(df.entities[0]) 

In [15]:
a

[('GPE', '烏克蘭'),
 ('ORG', '外交部'),
 ('TIME', '傍晚'),
 ('CARDINAL', '4000'),
 ('ORG', '外交部長'),
 ('PERSON', '吳釗燮'),
 ('TIME', '中午'),
 ('ORG', '外交部'),
 ('TIME', '晚間'),
 ('ORG', '外交部'),
 ('DATE', '7日'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('CARDINAL', '1730'),
 ('CARDINAL', '4000'),
 ('DATE', '18日'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('PERSON', '吳釗燮'),
 ('DATE', '今天'),
 ('TIME', '中午'),
 ('ORG', '外交部'),
 ('PERSON', '吳釗燮'),
 ('ORG', '慈濟'),
 ('NORP', '烏克蘭'),
 ('PERSON', '吳釗燮'),
 ('ORG', '外交部'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('DATE', '6個月'),
 ('DATE', '3月18日'),
 ('TIME', '下午'),
 ('TIME', '5時'),
 ('CARDINAL', '20'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部')]

In [16]:
a += eval(df.entities[1]) 

In [17]:
a

[('GPE', '烏克蘭'),
 ('ORG', '外交部'),
 ('TIME', '傍晚'),
 ('CARDINAL', '4000'),
 ('ORG', '外交部長'),
 ('PERSON', '吳釗燮'),
 ('TIME', '中午'),
 ('ORG', '外交部'),
 ('TIME', '晚間'),
 ('ORG', '外交部'),
 ('DATE', '7日'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('CARDINAL', '1730'),
 ('CARDINAL', '4000'),
 ('DATE', '18日'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('PERSON', '吳釗燮'),
 ('DATE', '今天'),
 ('TIME', '中午'),
 ('ORG', '外交部'),
 ('PERSON', '吳釗燮'),
 ('ORG', '慈濟'),
 ('NORP', '烏克蘭'),
 ('PERSON', '吳釗燮'),
 ('ORG', '外交部'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('DATE', '6個月'),
 ('DATE', '3月18日'),
 ('TIME', '下午'),
 ('TIME', '5時'),
 ('CARDINAL', '20'),
 ('ORG', '外交部'),
 ('ORG', '外交部'),
 ('NORP', '烏克蘭'),
 ('ORG', '外交部'),
 ('DATE', '今天'),
 ('QUANTITY', '2000呎'),
 ('FAC', '新竹空軍基地'),
 ('CARDINAL', '2000'),
 ('TIME', '上午'),
 ('GPE', '台東'),
 ('PERSON', '黃重凱'),
 ('PERSON', '柳惠千'),
 ('TIME', '下午'),
 ('QUANTITY', '2000呎'),
 ('QUANTITY', '2000呎'),
 ('PERSON', '朱冠甍'),
 ('DATE', '民國109年'),
 ('PERSON', '許

## Count top 200 word frequency for each category given a certain NE, e.g., "PERSON"

    {'政治': [('黃重凱', 24),
    ('柳惠千', 13),
    ('喬建中', 12),
    ('徐國勇', 11),
    ('季欽', 8),
    ('許舒博', 7),
    ('游錫堃', 6),

In [18]:
news_links =['aipl', 'ait', 'aspt', 'asc', 'aie', 'amov','ahel','aopl','asoc','acul','acn']
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']

In [19]:
allowedNE=['PERSON']

In [20]:
top_cate_ne_words={} # result we want to get
ne_all=[]
for category in news_categories:
    df_group = df[df.category == category]  # 20 pieces
    ne_group = []

    # concatenate terms in a category
    for ne in df_group.entities:
        ne_group += eval(ne)

    # concatenate all terms
    ne_all += ne_group

    # Get top words by calling ne_word_frequency() function
    topwords = ne_word_frequency( ne_group )
    top_cate_ne_words[category]=topwords

# Process category '全部'
top_ne_all = ne_word_frequency(ne_all)
top_cate_ne_words['全部']=top_ne_all

In [21]:
top_cate_ne_words

{'政治': [('黃重凱', 24),
  ('柳惠千', 13),
  ('喬建中', 12),
  ('徐國勇', 11),
  ('季欽', 8),
  ('許舒博', 7),
  ('游錫堃', 6),
  ('侯富議', 6),
  ('蘇貞昌', 5),
  ('吳釗燮', 4),
  ('許佑格', 4),
  ('范惠君', 4),
  ('歐布萊恩', 4),
  ('陳其邁', 3),
  ('柯文哲', 3),
  ('陳智菡', 3),
  ('黃鼎翔', 3),
  ('王同義', 3),
  ('王美惠', 3),
  ('許邁德', 2)],
 '科技': [('林柏亨', 6),
  ('吳偉仁', 5),
  ('包淳偉', 4),
  ('黃晨瑋', 4),
  ('廖俊智', 4),
  ('波博', 3),
  ('黃文山', 3),
  ('艾偉', 3),
  ('王金燦', 2),
  ('庫克', 2),
  ('曾信超', 2),
  ('波皮斯庫', 2),
  ('王兆璋', 1),
  ('希塔拉姆', 1),
  ('吳宗信', 1),
  ('葉均蔚', 1),
  ('楊勇', 1),
  ('陳信安', 1),
  ('Tim哥', 1),
  ('夏哈', 1)],
 '運動': [('林昀儒', 24),
  ('鄭怡靜', 24),
  ('莊智淵', 17),
  ('陳將双', 12),
  ('潘武雄', 10),
  ('陳建安', 8),
  ('布雷迪', 7),
  ('布魯娜', 6),
  ('賈奈特', 6),
  ('艾倫', 6),
  ('塞爾蒂克', 5),
  ('李恩芯', 5),
  ('王志庭', 4),
  ('施冠宇', 4),
  ('加奇尼', 4),
  ('樂天', 3),
  ('安宰賢', 3),
  ('巴特瑞', 3),
  ('梁家榮', 3),
  ('黃子鵬', 3)],
 '證卷': [('林耕億', 3), ('葉獻文', 2), ('李昌鴻', 2), ('黃水可', 1)],
 '產經': [('陳耀祥', 17),
  ('朱澤民', 9),
  ('曾博昇', 6),
  ('杜書全', 5),
  ('蘇松輝', 4),

In [22]:
## To conveniently save data using pandas, we should convert dict to list.
list(top_cate_ne_words.items())

[('政治',
  [('黃重凱', 24),
   ('柳惠千', 13),
   ('喬建中', 12),
   ('徐國勇', 11),
   ('季欽', 8),
   ('許舒博', 7),
   ('游錫堃', 6),
   ('侯富議', 6),
   ('蘇貞昌', 5),
   ('吳釗燮', 4),
   ('許佑格', 4),
   ('范惠君', 4),
   ('歐布萊恩', 4),
   ('陳其邁', 3),
   ('柯文哲', 3),
   ('陳智菡', 3),
   ('黃鼎翔', 3),
   ('王同義', 3),
   ('王美惠', 3),
   ('許邁德', 2)]),
 ('科技',
  [('林柏亨', 6),
   ('吳偉仁', 5),
   ('包淳偉', 4),
   ('黃晨瑋', 4),
   ('廖俊智', 4),
   ('波博', 3),
   ('黃文山', 3),
   ('艾偉', 3),
   ('王金燦', 2),
   ('庫克', 2),
   ('曾信超', 2),
   ('波皮斯庫', 2),
   ('王兆璋', 1),
   ('希塔拉姆', 1),
   ('吳宗信', 1),
   ('葉均蔚', 1),
   ('楊勇', 1),
   ('陳信安', 1),
   ('Tim哥', 1),
   ('夏哈', 1)]),
 ('運動',
  [('林昀儒', 24),
   ('鄭怡靜', 24),
   ('莊智淵', 17),
   ('陳將双', 12),
   ('潘武雄', 10),
   ('陳建安', 8),
   ('布雷迪', 7),
   ('布魯娜', 6),
   ('賈奈特', 6),
   ('艾倫', 6),
   ('塞爾蒂克', 5),
   ('李恩芯', 5),
   ('王志庭', 4),
   ('施冠宇', 4),
   ('加奇尼', 4),
   ('樂天', 3),
   ('安宰賢', 3),
   ('巴特瑞', 3),
   ('梁家榮', 3),
   ('黃子鵬', 3)]),
 ('證卷', [('林耕億', 3), ('葉獻文', 2), ('李昌鴻', 2), ('黃水可', 1)]),
 ('產經

## All-in-one function

In [25]:
# All-in-one function
def get_top_ner_words():
    top_cate_ner_words={}
    words_all=[]
    for category in news_categories:
        df_group = df[df.category == category]
        words_group = []

        # concatenate terms in a category
        for row in df_group.entities:
            words_group += eval(row)

        # concatenate all terms
        words_all += words_group

        # Get top words by calling ne_word_frequency() function
        topwords = ne_word_frequency( words_group )
        top_cate_ner_words[category] = topwords

    topwords_all = ne_word_frequency(words_all)
    top_cate_ner_words['全部'] = topwords_all
    
    return list(top_cate_ner_words.items()) # convert to list
    # return top_cate_ne_words

In [26]:
allowedNE=['PERSON']
result = get_top_ner_words()

In [27]:
result

[('政治',
  [('黃重凱', 24),
   ('柳惠千', 13),
   ('喬建中', 12),
   ('徐國勇', 11),
   ('季欽', 8),
   ('許舒博', 7),
   ('游錫堃', 6),
   ('侯富議', 6),
   ('蘇貞昌', 5),
   ('吳釗燮', 4),
   ('許佑格', 4),
   ('范惠君', 4),
   ('歐布萊恩', 4),
   ('陳其邁', 3),
   ('柯文哲', 3),
   ('陳智菡', 3),
   ('黃鼎翔', 3),
   ('王同義', 3),
   ('王美惠', 3),
   ('許邁德', 2)]),
 ('科技',
  [('林柏亨', 6),
   ('吳偉仁', 5),
   ('包淳偉', 4),
   ('黃晨瑋', 4),
   ('廖俊智', 4),
   ('波博', 3),
   ('黃文山', 3),
   ('艾偉', 3),
   ('王金燦', 2),
   ('庫克', 2),
   ('曾信超', 2),
   ('波皮斯庫', 2),
   ('王兆璋', 1),
   ('希塔拉姆', 1),
   ('吳宗信', 1),
   ('葉均蔚', 1),
   ('楊勇', 1),
   ('陳信安', 1),
   ('Tim哥', 1),
   ('夏哈', 1)]),
 ('運動',
  [('林昀儒', 24),
   ('鄭怡靜', 24),
   ('莊智淵', 17),
   ('陳將双', 12),
   ('潘武雄', 10),
   ('陳建安', 8),
   ('布雷迪', 7),
   ('布魯娜', 6),
   ('賈奈特', 6),
   ('艾倫', 6),
   ('塞爾蒂克', 5),
   ('李恩芯', 5),
   ('王志庭', 4),
   ('施冠宇', 4),
   ('加奇尼', 4),
   ('樂天', 3),
   ('安宰賢', 3),
   ('巴特瑞', 3),
   ('梁家榮', 3),
   ('黃子鵬', 3)]),
 ('證卷', [('林耕億', 3), ('葉獻文', 2), ('李昌鴻', 2), ('黃水可', 1)]),
 ('產經

## Popular person

In [28]:
allowedNE=['PERSON']
get_top_ner_words()

[('政治',
  [('黃重凱', 24),
   ('柳惠千', 13),
   ('喬建中', 12),
   ('徐國勇', 11),
   ('季欽', 8),
   ('許舒博', 7),
   ('游錫堃', 6),
   ('侯富議', 6),
   ('蘇貞昌', 5),
   ('吳釗燮', 4),
   ('許佑格', 4),
   ('范惠君', 4),
   ('歐布萊恩', 4),
   ('陳其邁', 3),
   ('柯文哲', 3),
   ('陳智菡', 3),
   ('黃鼎翔', 3),
   ('王同義', 3),
   ('王美惠', 3),
   ('許邁德', 2)]),
 ('科技',
  [('林柏亨', 6),
   ('吳偉仁', 5),
   ('包淳偉', 4),
   ('黃晨瑋', 4),
   ('廖俊智', 4),
   ('波博', 3),
   ('黃文山', 3),
   ('艾偉', 3),
   ('王金燦', 2),
   ('庫克', 2),
   ('曾信超', 2),
   ('波皮斯庫', 2),
   ('王兆璋', 1),
   ('希塔拉姆', 1),
   ('吳宗信', 1),
   ('葉均蔚', 1),
   ('楊勇', 1),
   ('陳信安', 1),
   ('Tim哥', 1),
   ('夏哈', 1)]),
 ('運動',
  [('林昀儒', 24),
   ('鄭怡靜', 24),
   ('莊智淵', 17),
   ('陳將双', 12),
   ('潘武雄', 10),
   ('陳建安', 8),
   ('布雷迪', 7),
   ('布魯娜', 6),
   ('賈奈特', 6),
   ('艾倫', 6),
   ('塞爾蒂克', 5),
   ('李恩芯', 5),
   ('王志庭', 4),
   ('施冠宇', 4),
   ('加奇尼', 4),
   ('樂天', 3),
   ('安宰賢', 3),
   ('巴特瑞', 3),
   ('梁家榮', 3),
   ('黃子鵬', 3)]),
 ('證卷', [('林耕億', 3), ('葉獻文', 2), ('李昌鴻', 2), ('黃水可', 1)]),
 ('產經

## Save top-200 hot persons for future usage

We have already done this in our previous app.

In [29]:
import pandas as pd

In [30]:
allowedNE=['PERSON']
hotPersons = get_top_ner_words()

In [31]:
hotPersons

[('政治',
  [('黃重凱', 24),
   ('柳惠千', 13),
   ('喬建中', 12),
   ('徐國勇', 11),
   ('季欽', 8),
   ('許舒博', 7),
   ('游錫堃', 6),
   ('侯富議', 6),
   ('蘇貞昌', 5),
   ('吳釗燮', 4),
   ('許佑格', 4),
   ('范惠君', 4),
   ('歐布萊恩', 4),
   ('陳其邁', 3),
   ('柯文哲', 3),
   ('陳智菡', 3),
   ('黃鼎翔', 3),
   ('王同義', 3),
   ('王美惠', 3),
   ('許邁德', 2)]),
 ('科技',
  [('林柏亨', 6),
   ('吳偉仁', 5),
   ('包淳偉', 4),
   ('黃晨瑋', 4),
   ('廖俊智', 4),
   ('波博', 3),
   ('黃文山', 3),
   ('艾偉', 3),
   ('王金燦', 2),
   ('庫克', 2),
   ('曾信超', 2),
   ('波皮斯庫', 2),
   ('王兆璋', 1),
   ('希塔拉姆', 1),
   ('吳宗信', 1),
   ('葉均蔚', 1),
   ('楊勇', 1),
   ('陳信安', 1),
   ('Tim哥', 1),
   ('夏哈', 1)]),
 ('運動',
  [('林昀儒', 24),
   ('鄭怡靜', 24),
   ('莊智淵', 17),
   ('陳將双', 12),
   ('潘武雄', 10),
   ('陳建安', 8),
   ('布雷迪', 7),
   ('布魯娜', 6),
   ('賈奈特', 6),
   ('艾倫', 6),
   ('塞爾蒂克', 5),
   ('李恩芯', 5),
   ('王志庭', 4),
   ('施冠宇', 4),
   ('加奇尼', 4),
   ('樂天', 3),
   ('安宰賢', 3),
   ('巴特瑞', 3),
   ('梁家榮', 3),
   ('黃子鵬', 3)]),
 ('證卷', [('林耕億', 3), ('葉獻文', 2), ('李昌鴻', 2), ('黃水可', 1)]),
 ('產經

In [32]:
df_hotPersons = pd.DataFrame(hotPersons, columns = ['category','top_keys'])

In [33]:
df_hotPersons

,category,top_keys
0,政治,"[(黃重凱, 24), (柳惠千, 13), (喬建中, 12), (徐國勇, 11), (..."
1,科技,"[(林柏亨, 6), (吳偉仁, 5), (包淳偉, 4), (黃晨瑋, 4), (廖俊智,..."
2,運動,"[(林昀儒, 24), (鄭怡靜, 24), (莊智淵, 17), (陳將双, 12), (..."
3,證卷,"[(林耕億, 3), (葉獻文, 2), (李昌鴻, 2), (黃水可, 1)]"
4,產經,"[(陳耀祥, 17), (朱澤民, 9), (曾博昇, 6), (杜書全, 5), (蘇松輝..."
5,娛樂,"[(唐川, 20), (歸亞蕾, 18), (炎亞綸, 11), (威廉赫特, 10), (..."
6,生活,"[(陳時中, 13), (彭佳偉, 13), (黃恩鴻, 5), (李宗宏, 3), (江文..."
7,國際,"[(蘇利文, 5), (楊潔篪, 4), (阿布拉莫維奇, 3), (蒲亭, 3), (米佐..."
8,社會,"[(戴寧, 8), (孫道存, 7), (鄭文燦, 6), (孫芸芸, 4), (孫瑩瑩, ..."
9,文化,"[(吳密察, 15), (平珩, 10), (吳蠻, 7), (潘孟安, 6), (吳思瑤,..."


In [35]:
# df_hotPersons.to_csv('news_top_person_by_category_via_ner.csv', sep=',', index=False) # 與上周的任務相同

## Popular products

In [34]:
allowedNE=['PRODUCT']
get_top_ner_words()

[('政治',
  [('2000', 3),
   ('殲16戰機', 2),
   ('2017', 1),
   ('殲10戰機', 1),
   ('ADIZ', 1),
   ('GBU-10', 1),
   ('GBU-12', 1),
   ('Mk 82', 1),
   ('F-16', 1),
   ('2058', 1)]),
 ('科技',
  [('iPhone SE', 8),
   ('iPad Air', 8),
   ('M1 Ultra', 6),
   ('iPhone', 5),
   ('Mac Studio', 5),
   ('LINE', 5),
   ('SXSW 2022', 4),
   ('M1 Max', 4),
   ('Touch ID', 4),
   ('iPhone 13', 4),
   ('5G', 3),
   ('新iPad Air', 3),
   ('iOS 15.4', 3),
   ('Face ID', 3),
   ('A15', 3),
   ('Google', 3),
   ('HTC VIVE', 2),
   ('MyndVR', 2),
   ('Daisy', 2),
   ('Studio Display', 2)]),
 ('運動', []),
 ('證卷',
  [('波音777-300ER客機', 1), ('777F貨機', 1), ('777-300ER', 1), ('COVID-19', 1)]),
 ('產經', [('COVID-19', 2)]),
 ('娛樂', [('KKTV', 1)]),
 ('生活', []),
 ('國際',
  [('F-35', 3),
   ('F-18', 2),
   ('Lockh', 1),
   ('Tornado）', 1),
   ('Annegr', 1),
   ('波音', 1),
   ('龍捲風戰機', 1),
   ('龍捲風', 1)]),
 ('社會', []),
 ('文化', []),
 ('兩岸', [('COVID-19', 1)]),
 ('全部',
  [('iPhone SE', 8),
   ('iPad Air', 8),
   ('M1 Ultra', 6),

In [35]:
allowedNE=['LAW']
get_top_ner_words()

[('政治', []),
 ('科技', [('LINE', 1)]),
 ('運動', []),
 ('證卷', []),
 ('產經', []),
 ('娛樂', []),
 ('生活', []),
 ('國際', [('國安法', 2), ('北約憲章第5條', 1)]),
 ('社會', []),
 ('文化', []),
 ('兩岸', [('香港國安法', 1), ('外國公司問責法', 1)]),
 ('全部',
  [('國安法', 2), ('LINE', 1), ('北約憲章第5條', 1), ('香港國安法', 1), ('外國公司問責法', 1)])]

## Geting your popular X

## We can get top 200 word frequecy for each news categroy by using "entities" column.

We can get top 200 word frequency for each category, if we includes all NEs.

We have done the same task in the app_top_keyword. We count the top keywords from "pos" column.

In [36]:
allowedNE=['EVENT','FAC','GPE','LANGUAGE','LAW','LOC','NORP','ORG','PERSON','PRODUCT','WORK_OF_ART']
top_group_ner_words = get_top_ner_words()

In [37]:
top_group_ner_words

[('政治',
  [('台灣', 25),
   ('黃重凱', 24),
   ('外交部', 16),
   ('柳惠千', 13),
   ('立法院', 13),
   ('喬建中', 12),
   ('烏克蘭', 11),
   ('徐國勇', 11),
   ('台東', 10),
   ('監察院', 9),
   ('法國', 8),
   ('行政院', 8),
   ('季欽', 8),
   ('台東志航基地', 8),
   ('國防部', 7),
   ('許舒博', 7),
   ('游錫堃', 6),
   ('台東馬偕醫院', 6),
   ('侯富議', 6),
   ('內政部', 6)]),
 ('科技',
  [('台灣', 24),
   ('蘋果', 20),
   ('俄羅斯', 13),
   ('iPhone SE', 8),
   ('iPad Air', 8),
   ('LINE', 8),
   ('烏克蘭', 7),
   ('南極', 7),
   ('中研院', 6),
   ('M1 Ultra', 6),
   ('林柏亨', 6),
   ('印度', 5),
   ('中國', 5),
   ('iPhone', 5),
   ('吳偉仁', 5),
   ('月球', 5),
   ('Mac Studio', 5),
   ('德國', 5),
   ('SXSW 2022', 4),
   ('美國', 4)]),
 ('運動',
  [('林昀儒', 24),
   ('鄭怡靜', 24),
   ('莊智淵', 17),
   ('台灣', 13),
   ('陳將双', 12),
   ('獅隊', 11),
   ('新加坡', 10),
   ('潘武雄', 10),
   ('桃猿隊', 9),
   ('陳建安', 8),
   ('布雷迪', 7),
   ('巴西', 6),
   ('布魯娜', 6),
   ('兄弟', 6),
   ('賈奈特', 6),
   ('艾倫', 6),
   ('中央社', 5),
   ('印度', 5),
   ('桃猿', 5),
   ('施冠宇', 5)]),
 ('證卷',
  [('台灣', 18),
   ('台股

In [38]:
df_top_group_ner_words = pd.DataFrame(top_group_ner_words, columns = ['category','top_keys'])

In [39]:
df_top_group_ner_words

,category,top_keys
0,政治,"[(台灣, 25), (黃重凱, 24), (外交部, 16), (柳惠千, 13), (立..."
1,科技,"[(台灣, 24), (蘋果, 20), (俄羅斯, 13), (iPhone SE, 8)..."
2,運動,"[(林昀儒, 24), (鄭怡靜, 24), (莊智淵, 17), (台灣, 13), (陳..."
3,證卷,"[(台灣, 18), (台股, 15), (陽明, 9), (揚弈科技, 8), (友通, ..."
4,產經,"[(台灣, 22), (陳耀祥, 17), (中國, 12), (歐盟, 12), (歐洲,..."
5,娛樂,"[(唐川, 20), (歸亞蕾, 20), (台灣, 11), (美國, 11), (炎亞綸..."
6,生活,"[(陳時中, 13), (彭佳偉, 13), (台電, 9), (台中一中, 6), (黃恩..."
7,國際,"[(俄國, 24), (俄羅斯, 23), (烏克蘭, 16), (香港, 14), (印度..."
8,社會,"[(消防局, 9), (戴寧, 8), (環保局, 8), (孫道存, 7), (台南高分院..."
9,文化,"[(台灣, 36), (故宮, 20), (吳密察, 15), (高雄, 14), (平珩,..."


In [42]:
# df_top_group_ner_words.to_csv('news_topkey_with_category_via_ner.csv', sep=',', index=False) # 與第一次的Django任務相同

# Count top-200 hot keywords for each NE name

count keywords for each NE name

In [40]:
# NER詞性
NE_Name=['CARDINAL','DATE','EVENT','FAC','GPE','LANGUAGE','LAW','LOC','MONEY','NORP','ORDINAL','ORG','PERCENT','PERSON','PRODUCT','QUANTITY','TIME','WORK_OF_ART']

In [41]:
%%time
# It takes time, at least 2 minitues.
top_word_NE=[]
for ne_name in NE_Name:
    allowedNE= [ne_name]
    topwords = get_top_ner_words()
    top_word_NE.append([ne_name, topwords])

CPU times: total: 2.16 s
Wall time: 2.4 s


In [42]:
top_word_NE

[['CARDINAL',
  [('政治',
    [('2000', 7),
     ('4000', 2),
     ('13', 2),
     ('54', 2),
     ('1730', 1),
     ('20', 1),
     ('38', 1),
     ('16', 1),
     ('39', 1),
     ('30', 1),
     ('90', 1),
     ('10萬', 1),
     ('12', 1),
     ('11', 1),
     ('41', 1),
     ('2017', 1),
     ('2036', 1)]),
   ('科技',
    [('4.0', 4),
     ('1200萬', 3),
     ('23', 2),
     ('120萬', 2),
     ('之一', 2),
     ('64', 2),
     ('25', 1),
     ('200', 1),
     ('15', 1),
     ('1萬', 1),
     ('2.5', 1),
     ('4000', 1),
     ('20', 1),
     ('60', 1),
     ('5G', 1),
     ('30', 1),
     ('33', 1),
     ('1176', 1),
     ('一半', 1),
     ('19', 1)]),
   ('運動',
    [('16', 19),
     ('11', 19),
     ('10', 5),
     ('32', 3),
     ('22', 2),
     ('57', 2),
     ('12', 2),
     ('7萬', 1),
     ('103萬', 1),
     ('31', 1),
     ('91', 1),
     ('128', 1),
     ('84', 1),
     ('92', 1),
     ('95', 1),
     ('四分', 1),
     ('20', 1),
     ('48', 1),
     ('18', 1),
     ('21', 1)]),
   ('證卷',


## Save each ne's top 200 category word frequency

In [43]:
import pandas as pd

In [44]:
df_top_word_NE = pd.DataFrame(top_word_NE, columns = ['ne_name','top_keys'])

In [45]:
df_top_word_NE

,ne_name,top_keys
0,CARDINAL,"[(政治, [('2000', 7), ('4000', 2), ('13', 2), ('..."
1,DATE,"[(政治, [('今天', 35), ('12天', 2), ('5月', 2), ('同年..."
2,EVENT,"[(政治, [('COVID-19（2019冠狀病毒疾病）疫情', 1), ('俄烏戰爭',..."
3,FAC,"[(政治, [('台東志航基地', 8), ('台東馬偕醫院', 5), ('新竹空軍基地'..."
4,GPE,"[(政治, [('台灣', 25), ('台東', 10), ('法國', 8), ('烏克..."
5,LANGUAGE,"[(政治, [('英文', 1)]), (科技, []), (運動, [('中文', 1)]..."
6,LAW,"[(政治, []), (科技, [('LINE', 1)]), (運動, []), (證卷,..."
7,LOC,"[(政治, [('台灣西南防空識別區', 1), ('北美', 1), ('屯稜線', 1)..."
8,MONEY,"[(政治, [('新台幣6億元', 1), ('3萬5703美元', 1), ('約101萬..."
9,NORP,"[(政治, [('烏克蘭', 6), ('德商', 2), ('歐商', 1), ('美商'..."


In [49]:
df_top_word_NE.to_csv('news_topkey_by_ner_and_category.csv', sep=',', index=False) # 本次的NER任務

# Another way to analyze by concatenating counter objects 

Code for your references

In [50]:
from collections import Counter
a = Counter({'menu': 20, 'good': 15, 'happy': 10, 'bar': 5})
b = Counter({'menu': 1, 'good': 1, 'bar': 3})
a + b

Counter({'menu': 21, 'good': 16, 'happy': 10, 'bar': 8})

In [51]:
top_cate_ner_words={} # final result
counter_all = Counter() # counter for category '全部'
for category in news_categories:
    
    df_group = df[df.category == category]
    
    # concatenate all filtered words in the same category
    words_group = []
    for row in df_group.entities:
        
        # filter words for each news
        filtered_words =[]
        for (ner, word) in eval(row):
            if (len(word) >= 2) & (ner in allowedNE):
                filtered_words.append(word)
                
        # concatenate filtered words  
        words_group += filtered_words

    # now we can count word frequency
    counter = Counter( words_group )
    
    # We use adding counter to get counter_all: 
    counter_all += counter

    topwords = counter.most_common(200)
    # store topwords
    top_cate_ner_words[category]= topwords

# Process category '全部'
top_cate_ne_words['全部'] = counter_all.most_common(200)

In [52]:
# All-in-one function
def get_top_ner_words():
    
    top_cate_ner_words={} # final result
    counter_all = Counter() # counter for category '全部'
    for category in news_categories:

        df_group = df[df.category == category]

        # concatenate all filtered words in the same category
        words_group = []
        for row in df_group.entities:

            # filter words for each news
            filtered_words =[]
            for (ner, word) in eval(row):
                if (len(word) >= 2) & (ner in allowedNE):
                    filtered_words.append(word)

            # concatenate filtered words  
            words_group += filtered_words

        # now we can count word frequency
        counter = Counter( words_group )

        # counter 
        counter_all += counter
        topwords = counter.most_common(200)

        # store topwords
        top_cate_ner_words[category]= topwords

    # Process category '全部'
    top_cate_ner_words['全部'] = counter_all.most_common(200)
    return list(top_cate_ner_words.items())

In [53]:
#allowedNE=['EVENT','FAC','GPE','LANGUAGE','LAW','LOC','NORP','ORG','PERSON','PRODUCT','WORK_OF_ART']
allowedNE=['PERSON']
get_top_ner_words()

[('政治',
  [('黃重凱', 24),
   ('柳惠千', 13),
   ('喬建中', 12),
   ('徐國勇', 11),
   ('季欽', 8),
   ('許舒博', 7),
   ('游錫堃', 6),
   ('侯富議', 6),
   ('蘇貞昌', 5),
   ('吳釗燮', 4),
   ('許佑格', 4),
   ('范惠君', 4),
   ('歐布萊恩', 4),
   ('陳其邁', 3),
   ('柯文哲', 3),
   ('陳智菡', 3),
   ('黃鼎翔', 3),
   ('王同義', 3),
   ('王美惠', 3),
   ('許邁德', 2),
   ('蔣萬安', 2),
   ('蔡練生', 2),
   ('沈榮津', 2),
   ('朱冠甍', 1),
   ('蘇紫雲', 1),
   ('蔣經國', 1),
   ('陳長江', 1),
   ('沈啟明', 1),
   ('周艷', 1),
   ('劉瑞寧', 1),
   ('賴新峰', 1),
   ('王奕然', 1),
   ('惠楊', 1),
   ('申濱', 1),
   ('李華', 1),
   ('李群', 1),
   ('施博祥', 1),
   ('蘇莉莉', 1),
   ('林栢梧', 1),
   ('葉義深', 1),
   ('蔡福松', 1),
   ('熊厚基', 1),
   ('陳元泰', 1),
   ('張仁澤', 1),
   ('黃重凱平', 1),
   ('葉宜津', 1),
   ('蔡崇義', 1),
   ('賴鼎銘', 1),
   ('王文淵', 1),
   ('洪孟楷', 1),
   ('吳彬福', 1),
   ('吳世峰', 1),
   ('江金亮', 1),
   ('劉浩帆', 1),
   ('張健翔', 1),
   ('何威克', 1),
   ('虞德忠', 1),
   ('劉永祥', 1),
   ('鄭育騰', 1),
   ('何子雨', 1)]),
 ('科技',
  [('林柏亨', 6),
   ('吳偉仁', 5),
   ('包淳偉', 4),
   ('黃晨瑋', 4),
   ('廖俊智', 4),
   ('波博